# Homer Transformer

Le but du projet est d'entraîner un Transformer, codé à la main avec PyTorch, pour qu'il réponde comme Homer
Simpson. On lui donne une réplique (par exemple `Marge: Tu as fini ?`) et il génère la réponse d'Homer.

J'ai choisi une architecture encodeur-décodeur, comme dans le Transformer d'origine : l'encodeur lit toute la
réplique d'un coup, et le décodeur écrit la réponse token par token en allant chercher l'information dans la
sortie de l'encodeur (cross-attention). Ça colle bien au problème, qui est de transformer une phrase en une autre.

Les dialogues des Simpsons ne contiennent qu'environ 18 000 réponses d'Homer. Ma première version, entraînée
uniquement dessus avec 107 millions de paramètres, apprenait les dialogues par cœur dès la 5e epoch et faisait
à peine des phrases. La version actuelle corrige ça de deux façons : un modèle beaucoup plus petit
(11,1 millions de paramètres) et un pré-entraînement sur des sous-titres de films avant le fine-tuning sur Homer.

## Installation

Le notebook tourne sur Google Colab avec un GPU T4 (Exécution > Modifier le type d'exécution). Cette cellule
installe `tiktoken`, monte Google Drive pour y sauvegarder les modèles (Colab efface tout à la déconnexion) et
télécharge les dialogues des Simpsons s'ils ne sont pas déjà là.

In [ ]:
import sys
import urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q tiktoken
    from google.colab import drive
    drive.mount("/content/drive")

dataset_file = Path("data/Simpsons/dialogues.jsonl")
if not dataset_file.exists():
    dataset_file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://huggingface.co/datasets/Romaindcrl/Simpsons/resolve/main/dialogues.jsonl"
    urllib.request.urlretrieve(url, dataset_file)
print("dataset :", dataset_file, f"({dataset_file.stat().st_size / 1e6:.1f} Mo)")

In [43]:
import json
import random
import time
from collections import Counter
from pathlib import Path

import pyarrow.parquet as pq
import tiktoken
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

In [ ]:
class Head(nn.Module):
    """Une tête d'attention : self-attention si context est None, cross-attention sinon."""

    def __init__(self, n_embd: int, head_size: int, dropout: float) -> None:
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, context=None):
        source = x if context is None else context

        q = self.query(x)  # les questions viennent toujours de x (celui qui lit)
        k = self.key(source)  # les étiquettes viennent de la source
        v = self.value(source)  # les informations aussi

        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5  # (B, T_x, T_source)
        if mask is not None:
            wei = wei.masked_fill(mask == 0, float("-inf"))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ v  # (B, T_x, head_size)


class MultiHeadAttention(nn.Module):
    """Plusieurs têtes en parallèle, concaténées puis projetées."""

    def __init__(self, n_embd: int, n_head: int, dropout: float) -> None:
        super().__init__()
        self.heads = nn.ModuleList(Head(n_embd, n_embd // n_head, dropout) for _ in range(n_head))
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, context=None):
        return self.dropout(self.proj(torch.cat([h(x, mask, context) for h in self.heads], dim=-1)))


class FeedForward(nn.Module):
    """MLP appliqué à chaque position indépendamment."""

    def __init__(self, n_embd: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    """Bloc Transformer pre-norm : attention puis feed-forward, avec résiduels."""

    def __init__(self, n_embd: int, n_head: int, dropout: float) -> None:
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        x = x + self.sa(self.ln1(x), mask)
        return x + self.ffwd(self.ln2(x))


class GPTLanguageModel(nn.Module):
    """Transformer générique : décodeur avec un masque causal, encodeur sans masque."""

    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float) -> None:
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(Block(n_embd, n_head, dropout) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        """Renvoie les logits `(B, T, vocab_size)`."""
        _, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x, mask)
        return self.lm_head(self.ln_f(x))

In [ ]:
def causal_mask(T: int, device=None) -> torch.Tensor:
    """(T, T) : chaque position ne voit qu'elle-même et le passé (décodeur)."""
    return torch.tril(torch.ones(T, T, device=device, dtype=torch.bool))


def padding_mask(pad: torch.Tensor) -> torch.Tensor:
    """pad : (B, T) avec 1 = vrai token, 0 = remplissage -> (B, 1, T) : personne ne regarde le remplissage."""
    return pad.bool()[:, None, :]


class Encoder(nn.Module):
    """Chaque position voit toute la séquence. Renvoie un vecteur par position."""

    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(Block(n_embd, n_head, dropout) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(n_embd)
        # pas de lm_head : l'encodeur ne prédit rien, il produit des représentations

    def forward(self, idx: torch.Tensor, pad: torch.Tensor | None = None) -> torch.Tensor:
        """idx : (B, T). pad optionnel : (B, T), 1 = vrai token, 0 = remplissage.
        Renvoie `(B, T, n_embd)`."""
        _, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))

        # pas de masque causal : seulement le padding, s'il y en a
        mask = padding_mask(pad) if pad is not None else None

        for block in self.blocks:
            x = block(x, mask)
        return self.ln_f(x)

In [ ]:
class DecoderBlock(nn.Module):
    """Bloc décodeur pre-norm : self-attention masquée, cross-attention, puis feed-forward."""

    def __init__(self, n_embd: int, n_head: int, dropout: float) -> None:
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, dropout)  # self-attention
        self.ca = MultiHeadAttention(n_embd, n_head, dropout)  # cross-attention
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, enc_out, self_mask=None, cross_mask=None):
        # 1. le texte du décodeur se regarde lui-même (sans voir le futur)
        x = x + self.sa(self.ln1(x), self_mask)
        # 2. le décodeur va chercher l'information dans la sortie de l'encodeur
        x = x + self.ca(self.ln2(x), cross_mask, context=enc_out)
        # 3. chaque position réfléchit seule
        return x + self.ffwd(self.ln3(x))

In [ ]:
# petit test des dimensions, en self-attention puis en cross-attention
mha = MultiHeadAttention(n_embd=32, n_head=4, dropout=0.0)
x = torch.randn(2, 5, 32)
enc = torch.randn(2, 9, 32)
print(mha(x).shape)
print(mha(x, context=enc).shape)

## Le décodeur et le modèle complet

Le décodeur empile des `DecoderBlock` et se termine par une `lm_head` qui donne un score à chaque token du
vocabulaire.

L'encodeur, le décodeur et la `lm_head` partagent une seule table d'embeddings (weight tying, comme dans GPT-2).
Les trois font le même travail dans un sens ou dans l'autre, token vers vecteur ou vecteur vers token. Dans la
première version, ces trois tables représentaient à elles seules 72 % des paramètres. Les partager divise cette
partie par trois, et chaque ligne est entraînée dès que son token apparaît, qu'il soit lu ou écrit.

Comme la table sert aussi à calculer les scores de sortie, je l'initialise avec de petites valeurs (std 0.02,
comme GPT-2). Avec l'initialisation par défaut de PyTorch (std 1), les scores du début sont énormes et la loss
de départ explose.

In [ ]:
class Decoder(nn.Module):
    """Écrit la réponse : chaque position ne voit que le passé et regarde la sortie de l'encodeur."""

    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(DecoderBlock(n_embd, n_head, dropout) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, enc_out, enc_pad=None):
        """idx : (B, T) la réponse en cours. enc_out : (B, T_src, n_embd). enc_pad : (B, T_src).
        Renvoie les logits `(B, T, vocab_size)`."""
        _, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))

        # Le padding de la réponse est toujours à la fin, le masque causal le cache déjà.
        # Par contre il faut masquer celui de la question pour la cross-attention.
        self_mask = causal_mask(T, idx.device)
        cross_mask = padding_mask(enc_pad) if enc_pad is not None else None

        for block in self.blocks:
            x = block(x, enc_out, self_mask, cross_mask)
        return self.lm_head(self.ln_f(x))


class HomerTransformer(nn.Module):
    """L'encodeur lit la réplique, le décodeur écrit la réponse."""

    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float) -> None:
        super().__init__()
        self.block_size = block_size
        self.encoder = Encoder(vocab_size, block_size, n_embd, n_head, n_layer, dropout)
        self.decoder = Decoder(vocab_size, block_size, n_embd, n_head, n_layer, dropout)

        # weight tying : une seule table pour l'encodeur, le décodeur et la lm_head
        self.decoder.token_embedding_table = self.encoder.token_embedding_table
        self.decoder.lm_head.weight = self.encoder.token_embedding_table.weight

        for table in (self.encoder.token_embedding_table,
                      self.encoder.position_embedding_table, self.decoder.position_embedding_table):
            nn.init.normal_(table.weight, std=0.02)

    def forward(self, src, src_pad, tgt_in):
        enc_out = self.encoder(src, src_pad)
        return self.decoder(tgt_in, enc_out, src_pad)

## Hyperparamètres

`N_EMBD = 256` avec 4 blocs donne 11,1 millions de paramètres. J'étais parti sur 512, mais avec aussi peu de
données le modèle apprenait par cœur. Les autres valeurs sont classiques pour un petit Transformer.

Le fine-tuning utilise un learning rate trois fois plus petit que le pré-entraînement, pour adapter le modèle
au style d'Homer sans lui faire oublier le français appris sur les sous-titres.

`PRETRAIN` et `FINETUNE` permettent de sauter une phase et de recharger le modèle déjà sauvegardé. C'est utile
sur Colab quand la session coupe au milieu, ou pour simplement discuter avec un modèle déjà entraîné.

In [ ]:
# Données
DATA_PATH = Path("data/Simpsons/dialogues.jsonl")
MAX_LEN = 64  # longueur max d'une réplique ou d'une réponse, en tokens
VAL_RATIO = 0.1

# Sous-titres pour le pré-entraînement (fichier de 200 Mo)
SUBTITLES_URL = ("https://huggingface.co/datasets/sentence-transformers/parallel-sentences-opensubtitles"
                 "/resolve/main/en-fr/train-00000-of-00002.parquet")
SUBTITLES_PATH = Path("data/opensubtitles_fr.parquet")
N_SUBTITLE_PAIRS = 300_000  # environ 2 Go de RAM par million de paires, Colab en a 12
MIN_TOKEN_COUNT = 5

# Modèle
N_EMBD = 256
N_HEAD = 4
N_LAYER = 4
DROPOUT = 0.1

# Entraînement
BATCH_SIZE = 64
PRETRAIN = False  # False : on recharge homer_pretrain.pt au lieu de refaire la phase 1
FINETUNE = False  # False : on garde homer_transformer.pt tel quel
PRETRAIN_EPOCHS = 3
PRETRAIN_LR = 3e-4
FINETUNE_EPOCHS = 5
FINETUNE_LR = 1e-4

# sur Colab les modèles vont dans le Drive, en local dans le dossier du projet
SAVE_DIR = Path("/content/drive/MyDrive") if IN_COLAB else Path(".")
PRETRAIN_PATH = SAVE_DIR / "homer_pretrain.pt"
CHECKPOINT_PATH = SAVE_DIR / "homer_transformer.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
random.seed(42)
USE_FP16 = device == "cuda"

print("device :", device)
if device == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

## Tokenizer

J'utilise le tokenizer BPE de GPT-2 (`tiktoken`) plutôt qu'un découpage par caractères : les séquences sont
beaucoup plus courtes et le modèle n'a pas à réapprendre l'orthographe de chaque mot. Il a été entraîné sur de
l'anglais, donc un mot français fait en moyenne deux tokens, mais ça reste raisonnable.

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

gpt2_ids = tokenizer.encode_ordinary("D'oh! Mmm... donuts")
print(gpt2_ids)
print([tokenizer.decode([i]) for i in gpt2_ids])

## Les dialogues des Simpsons

`dialogues.jsonl` contient un épisode par ligne. Dans le champ `text`, chaque ligne est une réplique
`Personnage: phrase`, et une ligne vide marque un changement de scène. Chaque fois qu'Homer répond à un autre
personnage dans la même scène, je garde la paire (réplique, réponse d'Homer).

Je laisse le nom du personnage dans l'entrée (`Marge: ...`) pour que le modèle sache à qui il répond. C'est aussi
ce qui permet de lui parler à la fin en se faisant passer pour Marge.

In [ ]:
def load_pairs(path: Path) -> list[tuple[str, str]]:
    """Renvoie les paires (réplique d'un personnage, réponse d'Homer)."""
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:  # une ligne = un épisode
            episode = json.loads(line)
            previous = None

            for replique in episode["text"].split("\n"):
                if ": " not in replique:  # ligne vide : nouvelle scène
                    previous = None
                    continue

                speaker, words = replique.split(": ", 1)
                speaker, words = speaker.strip(), words.strip()
                if speaker == "Homer" and words and previous and not previous.startswith("Homer:"):
                    pairs.append((previous, words))
                previous = f"{speaker}: {words}"
    return pairs


pairs = load_pairs(DATA_PATH)
print(f"{len(pairs)} paires chargées depuis {DATA_PATH}")
print(f"dont {sum(q.startswith('Marge:') for q, _ in pairs)} où Homer répond à Marge")
for question, answer in pairs[:3]:
    print(question, "  ->  Homer:", answer)

## Les sous-titres (pré-entraînement)

18 000 paires, c'est trop peu pour apprendre à faire des phrases. Avant le fine-tuning sur Homer, je pré-entraîne
donc le modèle sur des sous-titres de films et de séries en français, tirés du corpus OpenSubtitles
([Hugging Face](https://huggingface.co/datasets/sentence-transformers/parallel-sentences-opensubtitles),
source [OPUS](https://opus.nlpl.eu/)). C'est du français parlé et familier, proche du ton des Simpsons, et les
répliques sont dans l'ordre des dialogues.

Chaque réplique est associée à celle qui la suit. Il n'y a pas de séparation entre les films, donc quelques
paires mélangent deux scènes, mais pour apprendre le français ça ne pose pas de problème.

In [ ]:
if not SUBTITLES_PATH.exists():
    print("téléchargement des sous-titres (200 Mo)...")
    SUBTITLES_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(SUBTITLES_URL, SUBTITLES_PATH)

# la colonne "non_english" contient le sous-titre français, vide quand il n'y en a pas
lines = pq.read_table(SUBTITLES_PATH, columns=["non_english"]).column(0).to_pylist()


def subtitle_pairs(lines: list[str], n_pairs: int) -> list[tuple[str, str]]:
    """Paires (réplique, réplique suivante) à partir des sous-titres."""
    pairs = []
    for line, next_line in zip(lines, lines[1:]):
        line = line.strip().lstrip("- ")  # enlève les tirets de dialogue
        next_line = next_line.strip().lstrip("- ")
        if line and next_line:
            pairs.append((line, next_line))
        if len(pairs) == n_pairs:
            break
    return pairs


pretrain_pairs = subtitle_pairs(lines, N_SUBTITLE_PAIRS)
del lines
print(f"{len(pretrain_pairs)} paires de sous-titres")
for question, answer in pretrain_pairs[:3]:
    print(question, "  ->  ", answer)

### Le vocabulaire

GPT-2 connaît 50 257 tokens, mais la plupart n'apparaissent jamais dans nos données (code, autres langues, mots
anglais rares). Je ne garde que les tokens des Simpsons et ceux des sous-titres qui reviennent au moins 5 fois,
ce qui fait environ 14 000 tokens. La table d'embeddings est donc 3,5 fois plus petite. Les tokens plus rares
deviennent `UNK`.

Les numéros 0 à 3 sont réservés aux tokens spéciaux : `PAD` (remplissage), `BOS` (début de réponse), `EOS`
(fin de réponse) et `UNK`. `encode` et `decode` font la conversion entre les numéros GPT-2 et les nôtres.

In [ ]:
homer_tokens = {t for question, answer in pairs
                for t in tokenizer.encode_ordinary(question) + tokenizer.encode_ordinary(answer)}
subtitle_counts = Counter(t for question, answer in pretrain_pairs
                          for t in tokenizer.encode_ordinary(question) + tokenizer.encode_ordinary(answer))
frequent_subtitle_tokens = {t for t, count in subtitle_counts.items() if count >= MIN_TOKEN_COUNT}
gpt2_ids = sorted(homer_tokens | frequent_subtitle_tokens)

SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3


def build_vocab(gpt2_ids):
    """Tables de correspondance entre nos numéros et ceux de GPT-2."""
    to_gpt2 = [None] * len(SPECIAL_TOKENS) + list(gpt2_ids)
    from_gpt2 = {g: i for i, g in enumerate(to_gpt2) if g is not None}
    return to_gpt2, from_gpt2


to_gpt2, from_gpt2 = build_vocab(gpt2_ids)
VOCAB_SIZE = len(to_gpt2)


def encode(text: str) -> list[int]:
    return [from_gpt2.get(t, UNK_ID) for t in tokenizer.encode_ordinary(text)]


def decode(ids: list[int]) -> str:
    return tokenizer.decode([to_gpt2[i] for i in ids if i >= len(SPECIAL_TOKENS)])


print(f"vocabulaire : {VOCAB_SIZE} tokens (au lieu de {tokenizer.n_vocab + 2} avec tout GPT-2)")
ids = encode("Marge, je veux une bière.")
print(ids, "->", decode(ids))

### Dataset et DataLoader

Pour chaque paire, l'encodeur reçoit la réplique et le décodeur reçoit `[BOS] + réponse + [EOS]`. Pendant
l'entraînement, le décodeur prend `tgt[:-1]` en entrée et doit prédire `tgt[1:]`, c'est-à-dire le token suivant
à chaque position (teacher forcing). Toutes les positions sont calculées en parallèle grâce au masque causal.

`collate` complète les séquences d'un batch avec des `PAD`, et `make_loaders` sépare entraînement et validation.

In [ ]:
class DialogueDataset(Dataset):
    """Paires (réplique, réponse) tokenisées une seule fois au départ."""

    def __init__(self, pairs: list[tuple[str, str]]) -> None:
        self.examples = []
        for question, answer in pairs:
            src = encode(question)[:MAX_LEN]
            tgt = [BOS_ID] + encode(answer)[:MAX_LEN - 2] + [EOS_ID]
            self.examples.append((torch.tensor(src), torch.tensor(tgt)))

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


def collate(batch):
    srcs, tgts = zip(*batch)
    src = pad_sequence(srcs, batch_first=True, padding_value=PAD_ID)
    tgt = pad_sequence(tgts, batch_first=True, padding_value=PAD_ID)
    src_pad = (src != PAD_ID).long()  # 1 = vrai token, 0 = remplissage
    return src, src_pad, tgt


def make_loaders(pairs: list[tuple[str, str]]):
    pairs = pairs[:]
    random.shuffle(pairs)
    n_val = max(1, int(len(pairs) * VAL_RATIO))
    train_loader = DataLoader(DialogueDataset(pairs[n_val:]), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(DialogueDataset(pairs[:n_val]), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    print(f"train : {len(train_loader.dataset)} exemples, val : {len(val_loader.dataset)} exemples")
    return train_loader, val_loader


print("sous-titres :", end=" ")
pretrain_train_loader, pretrain_val_loader = make_loaders(pretrain_pairs)
print("Homer :", end=" ")
homer_train_loader, homer_val_loader = make_loaders(pairs)

src, src_pad, tgt = next(iter(homer_train_loader))
print("src", tuple(src.shape), "| tgt", tuple(tgt.shape))

## Le modèle

Un seul modèle passe par les deux phases. Ses réglages sont sauvegardés avec les poids, pour pouvoir le
reconstruire à l'identique au moment de le recharger.

In [ ]:
model_config = dict(
    vocab_size=VOCAB_SIZE, block_size=MAX_LEN, n_embd=N_EMBD,
    n_head=N_HEAD, n_layer=N_LAYER, dropout=DROPOUT,
)
model = HomerTransformer(**model_config).to(device)

# en float16 les très petits gradients tombent à 0, le scaler multiplie la loss pour l'éviter
scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params / 1e6:.1f} M paramètres")

## Fonctions d'entraînement

La loss est une cross-entropy sur le token suivant, en ignorant les positions de remplissage. Sur GPU, le
forward se fait en float16 (`torch.autocast`), ce qui va environ deux fois plus vite sur une T4 ; le
`GradScaler` évite que les petits gradients deviennent nuls. Je limite aussi la norme des gradients à 1 pour
qu'un batch atypique ne déstabilise pas l'entraînement.

`evaluate` calcule la même loss sur la validation, sans mettre à jour les poids. C'est elle qui sert à repérer
l'overfitting : quand elle remonte alors que la loss d'entraînement baisse encore, le modèle apprend par cœur.

In [ ]:
def compute_loss(model, batch):
    src, src_pad, tgt = (t.to(device) for t in batch)
    tgt_in = tgt[:, :-1]
    tgt_out = tgt[:, 1:]  # décalé d'un cran : on prédit le token suivant

    with torch.autocast(device_type=device, dtype=torch.float16, enabled=USE_FP16):
        logits = model(src, src_pad, tgt_in)
    return F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1), ignore_index=PAD_ID)


def train_one_epoch(model, loader, optimizer) -> float:
    model.train()
    total = 0.0
    for i, batch in enumerate(loader, 1):
        loss = compute_loss(model, batch)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)  # pour que le clipping voie les vrais gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
        if i % 500 == 0:
            print(f"   batch {i}/{len(loader)} | loss {total / i:.3f}")
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader) -> float:
    model.eval()
    total = 0.0
    for batch in loader:
        total += compute_loss(model, batch).item()
    return total / len(loader)

## Générer une réponse

La génération se fait token par token : on encode la réplique une fois, on part de `[BOS]` et on ajoute à chaque
tour le token choisi, jusqu'à `EOS`. Plutôt que de prendre toujours le token le plus probable (ce qui donne des
réponses répétitives), je tire au sort parmi les 50 plus probables (top-k), avec une température de 0.8 pour
rester assez prudent.

In [ ]:
@torch.no_grad()
def reply(model, question: str, max_new_tokens: int = 50, temperature: float = 0.8, top_k: int = 50) -> str:
    model.eval()
    src = torch.tensor([encode(question)[:MAX_LEN]], device=device)
    src_pad = torch.ones_like(src)
    enc_out = model.encoder(src, src_pad)

    out = [BOS_ID]
    for _ in range(min(max_new_tokens, MAX_LEN - 1)):
        idx = torch.tensor([out], device=device)
        logits = model.decoder(idx, enc_out, src_pad)[0, -1]
        logits[PAD_ID] = logits[BOS_ID] = logits[UNK_ID] = float("-inf")  # tokens qu'il ne doit jamais écrire

        logits = logits / temperature
        kth_best = torch.topk(logits, top_k).values[-1]
        logits[logits < kth_best] = float("-inf")
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()

        if next_id == EOS_ID:
            break
        out.append(next_id)
    return decode(out[1:])


print(reply(model, "Marge: Homer, tu as fini ton travail ?"))  # avant entraînement, la réponse est aléatoire

## Entraînement

`train` gère une phase complète. Après chaque epoch, le modèle est sauvegardé s'il a la meilleure loss de
validation. À la fin, on recharge ce meilleur modèle, pour que la phase suivante parte de lui et non de la
dernière epoch, qui peut déjà être en overfitting.

In [ ]:
TEST_QUESTION = "Marge: Homer, tu as fini ton travail ?"


def train(model, train_loader, val_loader, epochs: int, lr: float, save_path: Path) -> float:
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    best_val = float("inf")

    for epoch in range(1, epochs + 1):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_loss = evaluate(model, val_loader)

        saved = ""
        if val_loss < best_val:
            best_val = val_loss
            torch.save({"config": model_config, "vocab": gpt2_ids, "weights": model.state_dict()}, save_path)
            saved = " (sauvegardé)"

        print(f"epoch {epoch:3d} | train {train_loss:.3f} | val {val_loss:.3f} | {time.time() - start:.1f}s{saved}")
        print("   Homer :", reply(model, TEST_QUESTION))

    model.load_state_dict(torch.load(save_path, map_location=device)["weights"])
    return best_val

### Phase 1 : pré-entraînement sur les sous-titres

Avec `PRETRAIN = False`, on recharge `homer_pretrain.pt` au lieu de réentraîner. Le fichier doit avoir été créé
avec les mêmes réglages et le même vocabulaire, sinon la cellule s'arrête avec une erreur.

In [ ]:
if PRETRAIN:
    best = train(model, pretrain_train_loader, pretrain_val_loader, PRETRAIN_EPOCHS, PRETRAIN_LR, PRETRAIN_PATH)
    print(f"meilleure loss val (sous-titres) : {best:.3f}")
else:
    pretrained = torch.load(PRETRAIN_PATH, map_location=device)
    if pretrained["config"] != model_config or pretrained["vocab"] != gpt2_ids:
        raise ValueError(f"{PRETRAIN_PATH} a été entraîné avec d'autres réglages (taille du modèle ou vocabulaire). "
                         "Remets PRETRAIN = True pour refaire le pré-entraînement.")
    model.load_state_dict(pretrained["weights"])
    print("pré-entraînement repris depuis", PRETRAIN_PATH)

### Phase 2 : fine-tuning sur Homer

Même modèle, mais entraîné sur les réponses d'Homer avec un learning rate plus petit. Avec seulement 18 000
paires, la loss de validation remonte vite ; on garde de toute façon la meilleure epoch.

In [ ]:
if FINETUNE:
    best = train(model, homer_train_loader, homer_val_loader, FINETUNE_EPOCHS, FINETUNE_LR, CHECKPOINT_PATH)
    print(f"meilleure loss val (Homer) : {best:.3f}")

## Récupérer le modèle (Colab)

Le modèle final est dans le Drive (`homer_transformer.pt`). Cette cellule le télécharge pour pouvoir l'utiliser
en local.

In [ ]:
if IN_COLAB:
    if CHECKPOINT_PATH.exists():
        from google.colab import files
        files.download(str(CHECKPOINT_PATH))
    else:
        print("Pas encore de modèle : il est sauvegardé à la fin de la première epoch du fine-tuning.")

## Discuter avec Homer

On recharge le modèle final avec ses réglages et son vocabulaire, puis on lui parle en jouant Marge :
`parle_a_homer("...")` envoie une phrase précise, et la dernière cellule permet de discuter librement (dans
VS Code, la case de saisie apparaît en haut de la fenêtre). On arrête avec `quit` ou une ligne vide.

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
to_gpt2, from_gpt2 = build_vocab(checkpoint["vocab"])
model = HomerTransformer(**checkpoint["config"]).to(device)
model.load_state_dict(checkpoint["weights"])

for question in ["Marge: Homer, tu veux une salade ?", "Marge: Homer, la centrale a appelé.", "Marge: Tu m'aimes ?"]:
    print(question)
    print("Homer:", reply(model, question))
    print()

In [ ]:
def parle_a_homer(phrase: str) -> None:
    print("Marge:", phrase)
    print("Homer:", reply(model, f"Marge: {phrase}"))
    print()

In [ ]:
while True:
    phrase = input("Marge: ")
    if phrase.strip().lower() in ("", "quit"):
        break
    parle_a_homer(phrase)